In [ ]:
# Set DATA_ROOT (env var or config.py); all paths below derive from it
from config import DATA_ROOT


In [ ]:
import rasterio as rs
import numpy as np
import matplotlib.pyplot as plt
from rasterio.features import shapes
from shapely.geometry import shape
from matplotlib.colors import ListedColormap
import pickle
import os
from os.path import isdir, join, isfile
from rasterio.enums import Resampling
from matplotlib.colors import LogNorm
from sklearn.metrics import r2_score

In [ ]:
resolution = 100

In [ ]:
with rs.open(f'{DATA_ROOT}/Sumatra-AGB/pred_rasters/agbd_{resolution}m.tif', 'r') as src:
    sumatra_1 = src.read(1)
    target_crs = src.crs

## Agreement between GEDI L4B 1km and Sumatra 1km

In [ ]:
with rs.open(f'{DATA_ROOT}/Sumatra-AGB/pred_rasters/agbd_1000m.tif', 'r') as src:
    sumatra_1 = src.read(1)
    target_crs = src.crs

with rs.open(f'{DATA_ROOT}/EcosystemAnalysis/Models/Biomes/Sumatra/Data/GEDI_L4B_AGBD_Sumatra.tif', 'r') as src:
    gedi_1km = src.read(1)
    gedi_transform = src.transform
    gedi_crs = src.crs

# Reproject GEDI to Sumatra CRS
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.enums import Resampling
gedi_1km_reprojected = np.full(sumatra_1.shape, np.nan, dtype=np.float32)
transform, width, height = calculate_default_transform(
    gedi_crs, target_crs, src.width, src.height, *src.bounds)
reproject(
    source=gedi_1km,
    destination=gedi_1km_reprojected,
    src_transform=gedi_transform,
    src_crs=gedi_crs,
    dst_transform=transform,
    dst_crs=target_crs,
    resampling=Resampling.average)

print(gedi_1km.shape, gedi_1km_reprojected.shape, sumatra_1.shape)

In [ ]:
plt.imshow(gedi_1km, vmin=0.1, vmax=250, cmap='viridis')
plt.colorbar()
plt.show()
plt.imshow(gedi_1km_reprojected, vmin=0.1, vmax=250, cmap='viridis')
plt.colorbar()
plt.show()
plt.imshow(sumatra_1, vmin=0.1, vmax=250, cmap='viridis')
plt.colorbar()
plt.show()

In [ ]:
valid = ~np.isnan(sumatra_1) & (sumatra_1 > 0) & ~np.isnan(gedi_1km_reprojected) & (gedi_1km_reprojected > 0)

sumatra_values = sumatra_1[valid]
gedi_values = gedi_1km_reprojected[valid]

# Compute correlation metrics
from scipy.stats import pearsonr, spearmanr
pearson_corr, _ = pearsonr(gedi_values, sumatra_values)
spearman_corr, _ = spearmanr(gedi_values, sumatra_values)
print(f'Pearson correlation: {pearson_corr:.4f}')
print(f'Spearman correlation: {spearman_corr:.4f}')

# R2 score
from sklearn.metrics import r2_score
r2 = r2_score(gedi_values, sumatra_values)
print(f'R2 score: {r2:.4f}')

print(f'Number of valid pixels: {len(sumatra_values)}')

# density plot
plt.figure(figsize=(8,6))
plt.hist2d(sumatra_values, gedi_values, norm=LogNorm(), bins=100, cmap='viridis')
plt.colorbar()
plt.xlabel('Sumatra AGBD 1km (Mg/ha)')
plt.ylabel('GEDI L4B AGBD 1km (Mg/ha)')
plt.xlim(0, 250)
plt.ylim(0, 250)
plt.plot([0, 250], [0, 250], color='red', linestyle='--')

# Add a box with the correlation metrics
textstr = '\n'.join((
    f'Pearson: {pearson_corr:.3f}',
    f'Spearman: {spearman_corr:.3f}',
    f'$R^2$: {r2:.3f}',
))
props = dict(boxstyle='round', facecolor='white', alpha=0.8)
plt.text(0.73, 0.15, textstr, transform=plt.gca().transAxes, fontsize=10, verticalalignment='top', bbox=props)
plt.savefig('figs/agreement-gedi-sumatra-1km-density.png', dpi=800)
plt.show()

# Scatter plot
plt.figure(figsize=(8,6))
plt.scatter(sumatra_values, gedi_values, alpha=0.5, s=1)
plt.xlabel('Sumatra AGBD 1km (Mg/ha)')
plt.ylabel('GEDI L4B AGBD 1km (Mg/ha)')
plt.title('Scatter Plot of Sumatra AGBD vs GEDI L4B')
plt.xlim(0, 250)
plt.ylim(0, 250)
plt.plot([0, 250], [0, 250], color='red', linestyle='--')
plt.show()



## For our OG map, post-kriging on field

In [ ]:
# Best setting ########################
max_train_samples = 1000
stripe_size = 50
std_threshold = 100

In [ ]:
with rs.open('Data/merged_downsampled.tif', 'r') as src1, \
    rs.open(f'predictions/{max_train_samples}-{stripe_size}/corr_merged.tif', 'r') as src2, \
    rs.open(f'{DATA_ROOT}/Sumatra-AGB/pred_rasters/agbd_100m.tif', 'r') as src3 :
    
    pre_pred = src1.read(1)
    rescale_transform = src1.transform * src1.transform.scale((src1.width / pre_pred.shape[-1]), (src1.height / pre_pred.shape[-2]))

    post_pred = src2.read(1)
    post_std = src2.read(2)
    sumatra = src3.read(1)

    transform_corr_merged = src2.transform
    crs_corr_merged = src2.crs
    transform_sumatra = src3.transform
    crs_sumatra = src3.crs

In [ ]:
f, axarr = plt.subplots(1, 3, figsize=(15, 15), constrained_layout=True)
im1 = axarr[0].imshow(pre_pred, cmap='viridis', vmin=0, vmax=250)
axarr[0].set_title('Pre-Kriging AGB Density [t/ha]')
axarr[0].set_xticks([])  # Remove x-axis ticks
axarr[0].set_yticks([])  # Remove y-axis ticks
axarr[0].axis('off')
im2 = axarr[1].imshow(post_pred, cmap='viridis', vmin=0, vmax=250)
axarr[1].set_title('Post-Kriging AGB Density [t/ha]')
axarr[1].set_xticks([])  # Remove x-axis ticks
axarr[1].set_yticks([])  # Remove y-axis ticks
axarr[1].axis('off')
im3 = axarr[2].imshow(sumatra_1, cmap='viridis', vmin=0, vmax=250)
axarr[2].set_title('Reference AGB Density [t/ha]')
axarr[2].set_xticks([])  # Remove x-axis ticks
axarr[2].set_yticks([])  # Remove y-axis ticks
axarr[2].axis('off')

cbar = f.colorbar(im3, ax=axarr, shrink=0.2, orientation='vertical')
cbar.set_label('AGB Density [t/ha]')

#plt.savefig(f'{DATA_ROOT}/EcosystemAnalysis/Models/Biomes/plots/sumatra_kriging-pre-post.png', dpi=1200, bbox_inches='tight')
plt.show()

In [ ]:
f, axarr = plt.subplots(1, 2, figsize=(15, 15), constrained_layout=True)

im1 = axarr[0].imshow(post_pred, cmap='viridis', vmin=0, vmax=250)
axarr[0].set_title('Post-Kriging AGB Density [t/ha]')
axarr[0].set_xticks([])  # Remove x-axis ticks
axarr[0].set_yticks([])  # Remove y-axis ticks
axarr[0].axis('off')

# add colorbar for im1
cbar1 = f.colorbar(im1, ax=axarr[0], shrink=0.6, orientation='vertical')
cbar1.set_label('AGB Density [t/ha]')

im2 = axarr[1].imshow(post_std, cmap='viridis')
axarr[1].set_title('Post-Kriging Standard Deviation [t/ha]')
axarr[1].set_xticks([])  # Remove x-axis ticks
axarr[1].set_yticks([])  # Remove y-axis ticks
axarr[1].axis('off')

# add colorbar for im2
cbar2 = f.colorbar(im2, ax=axarr[1], shrink=0.6, orientation='vertical')
cbar2.set_label('AGB Standard Deviation [t/ha]')

plt.show()

Calculate metrics and plot results.

In [ ]:
# Load the test split
with open(f'splits/split_map_s={stripe_size}_t={std_threshold}.pkl', 'rb') as f: split_map = pickle.load(f)

# Evaluate on the test set
test_mask = (split_map == 3)

valid_mask = (~np.isnan(sumatra_1)) & (~np.isnan(pre_pred)) & (~np.isnan(post_pred))
test_mask = test_mask & valid_mask

pre_residuals = pre_pred[test_mask] - sumatra_1[test_mask]
post_residuals = post_pred[test_mask] - sumatra_1[test_mask]

print(f"RMSE before kriging: {np.sqrt(np.nanmean(pre_residuals**2)):.2f} [t/ha]")
print(f"RMSE after kriging: {np.sqrt(np.nanmean(post_residuals**2)):.2f} [t/ha]")

In [ ]:
# Binned residuas pre and post kriging

bins = np.arange(0, 300, 50)
lbs, ubs = bins[:-1], bins[1:]

pre_binned_residuals = [pre_residuals[(sumatra_1[test_mask] >= lb) & (sumatra_1[test_mask] < ub)] for lb, ub in zip(lbs, ubs)]
post_binned_residuals = [post_residuals[(sumatra_1[test_mask] >= lb) & (sumatra_1[test_mask] < ub)] for lb, ub in zip(lbs, ubs)]

fig, ax = plt.subplots(figsize=(10, 6))
box1 = ax.boxplot(pre_binned_residuals, showfliers=False, positions=bins[:-1] + 10, widths=20, patch_artist=True,
                  boxprops=dict(facecolor='lightblue', color='blue'),
                  medianprops=dict(color='darkblue'),
                  whiskerprops=dict(color='blue'),
                  capprops=dict(color='blue'),
                  flierprops=dict(color='blue', markeredgecolor='blue'))
box2 = ax.boxplot(post_binned_residuals, showfliers=False, positions=bins[:-1] + 30, widths=20, patch_artist=True,
                  boxprops=dict(facecolor='lightgreen', color='green'),
                  medianprops=dict(color='darkgreen'),
                  whiskerprops=dict(color='green'),
                  capprops=dict(color='green'),
                  flierprops=dict(color='green', markeredgecolor='green'))
ax.axhline(0, color='red', linestyle='--')
ax.set_xlabel('AGB Density Bins [t/ha]')
ax.set_ylabel('Residuals [t/ha]')
ax.set_title('Binned Residuals Before and After Kriging')
ax.legend([box1["boxes"][0], box2["boxes"][0]], ['Pre-Kriging', 'Post-Kriging'], loc='upper right')
#plt.savefig(f'{DATA_ROOT}/EcosystemAnalysis/Models/Biomes/plots/sumatra_kriging-binned-residuals.png', dpi=1200, bbox_inches='tight')
plt.show()

In [ ]:
# Density plots of residuals pre and post kriging

fig, axs = plt.subplots(1, 2, figsize=(18, 8))  # 1 row, 2 columns

# Shared max for limits
ma = np.nanmax(sumatra_1) + 50
step = 50
ticks = np.arange(0, ma + step, step)

# --- First plot ---
mask1 = (np.isnan(sumatra_1)) | (np.isnan(pre_pred))
h1 = axs[0].hist2d(sumatra_1[~mask1].squeeze(), pre_pred[~mask1].squeeze(), bins=(100, 100), cmap='Greens', norm=LogNorm())
cbar1 = fig.colorbar(h1[3], ax=axs[0])
cbar1.ax.tick_params(labelsize=14)
cbar1.ax.set_ylabel('Number of samples', fontsize=16)

axs[0].plot([0, ma], [0, ma], 'k--')
axs[0].set_xlim((0, ma))
axs[0].set_ylim((0, ma))
axs[0].set_xlabel('GEDI reference AGBD [Mg/ha]', fontsize=16)
axs[0].set_ylabel('Predicted AGB [Mg/ha]', fontsize=16)
axs[0].set_xticks(ticks)
axs[0].set_yticks(ticks)
axs[0].tick_params(labelsize=14)
axs[0].grid()
axs[0].set_aspect('equal')

# --- Second plot (example, change inputs as needed) ---
mask2 = (np.isnan(sumatra_1)) | (np.isnan(post_pred))
h2 = axs[1].hist2d(sumatra_1[~mask2].squeeze(), post_pred[~mask2].squeeze(), bins=(100, 100), cmap='Greens', norm=LogNorm())

cbar2 = fig.colorbar(h2[3], ax=axs[1])
cbar2.ax.tick_params(labelsize=14)
cbar2.ax.set_ylabel('Number of samples', fontsize=16)

axs[1].plot([0, ma], [0, ma], 'k--')
axs[1].set_xlim((0, ma))
axs[1].set_ylim((0, ma))
axs[1].set_xlabel('GEDI reference AGBD [Mg/ha]', fontsize=16)
axs[1].set_ylabel('Predicted AGB [Mg/ha]', fontsize=16)
axs[1].set_xticks(ticks)
axs[1].set_yticks(ticks)
axs[1].tick_params(labelsize=14)
axs[1].grid()
axs[1].set_aspect('equal')

plt.tight_layout()
#plt.savefig('vis/ours-density.png', dpi=1200)
plt.show()

## For the CCI map, post-kriging on field

In [ ]:
# Best setting ########################
max_train_samples = 1000
stripe_size = 50
std_threshold = 100

In [ ]:
with rs.open('Data/CCI_N00E100.tif', 'r') as src1, \
    rs.open(f'predictions/{max_train_samples}-{stripe_size}/CCI-corr_merged.tif', 'r') as src2, \
    rs.open(f'{DATA_ROOT}/Sumatra-AGB/pred_rasters/agbd_100m.tif', 'r') as src3 :
    
    pre_pred = src1.read(1, out_shape = (1, sumatra_1.shape[0], sumatra_1.shape[1]), resampling = Resampling.bilinear)
    rescale_transform = src1.transform * src1.transform.scale((src1.width / pre_pred.shape[-1]), (src1.height / pre_pred.shape[-2]))

    post_pred = src2.read(1)
    sumatra = src3.read(1)

    transform_corr_merged = src2.transform
    crs_corr_merged = src2.crs
    transform_sumatra = src3.transform
    crs_sumatra = src3.crs



Plot them.

In [ ]:
f, axarr = plt.subplots(1, 3, figsize=(15, 15), constrained_layout=True)
im1 = axarr[0].imshow(pre_pred, cmap='viridis', vmin=0, vmax=250)
axarr[0].set_title('Pre-Kriging AGB Density [t/ha]')
axarr[0].set_xticks([])  # Remove x-axis ticks
axarr[0].set_yticks([])  # Remove y-axis ticks
axarr[0].axis('off')

im2 = axarr[1].imshow(post_pred, cmap='viridis', vmin=0, vmax=250)
axarr[1].set_title('Post-Kriging AGB Density [t/ha]')
axarr[1].set_xticks([])  # Remove x-axis ticks
axarr[1].set_yticks([])  # Remove y-axis ticks
axarr[1].axis('off')

im3 = axarr[2].imshow(sumatra_1, cmap='viridis', vmin=0, vmax=250)
axarr[2].set_title('Reference AGB Density [t/ha]')
axarr[2].set_xticks([])  # Remove x-axis ticks
axarr[2].set_yticks([])  # Remove y-axis ticks
axarr[2].axis('off')

cbar = f.colorbar(im3, ax=axarr, shrink=0.2, orientation='vertical')
cbar.set_label('AGB Density [t/ha]')

#plt.tight_layout()
#plt.savefig(f'{DATA_ROOT}/EcosystemAnalysis/Models/Biomes/plots/sumatra_CCI-kriging-pre-post.png', dpi=1200, bbox_inches='tight')
plt.show()

Calculate metrics and plot results.

In [ ]:
# Load the test split
with open(f'splits/split_map_s={stripe_size}_t={std_threshold}.pkl', 'rb') as f: split_map = pickle.load(f)

# Evaluate on the test set
test_mask = (split_map == 3)

valid_mask = (~np.isnan(sumatra_1)) & (~np.isnan(pre_pred)) & (~np.isnan(post_pred))
test_mask = test_mask & valid_mask

pre_residuals = pre_pred[test_mask] - sumatra_1[test_mask]
post_residuals = post_pred[test_mask] - sumatra_1[test_mask]

print(f"RMSE before kriging: {np.sqrt(np.nanmean(pre_residuals**2)):.2f} [t/ha]")
print(f"RMSE after kriging: {np.sqrt(np.nanmean(post_residuals**2)):.2f} [t/ha]")

In [ ]:
# Binned residuas pre and post kriging

bins = np.arange(0, 300, 50)
lbs, ubs = bins[:-1], bins[1:]

pre_binned_residuals = [pre_residuals[(sumatra_1[test_mask] >= lb) & (sumatra_1[test_mask] < ub)] for lb, ub in zip(lbs, ubs)]
post_binned_residuals = [post_residuals[(sumatra_1[test_mask] >= lb) & (sumatra_1[test_mask] < ub)] for lb, ub in zip(lbs, ubs)]

fig, ax = plt.subplots(figsize=(10, 6))
box1 = ax.boxplot(pre_binned_residuals, showfliers=False, positions=bins[:-1] + 10, widths=20, patch_artist=True,
                  boxprops=dict(facecolor='lightblue', color='blue'),
                  medianprops=dict(color='darkblue'),
                  whiskerprops=dict(color='blue'),
                  capprops=dict(color='blue'),
                  flierprops=dict(color='blue', markeredgecolor='blue'))
box2 = ax.boxplot(post_binned_residuals, showfliers=False, positions=bins[:-1] + 30, widths=20, patch_artist=True,
                  boxprops=dict(facecolor='lightgreen', color='green'),
                  medianprops=dict(color='darkgreen'),
                  whiskerprops=dict(color='green'),
                  capprops=dict(color='green'),
                  flierprops=dict(color='green', markeredgecolor='green'))
ax.axhline(0, color='red', linestyle='--')
ax.set_xlabel('AGB Density Bins [t/ha]')
ax.set_ylabel('Residuals [t/ha]')
ax.set_title('Binned Residuals Before and After Kriging')
ax.legend([box1["boxes"][0], box2["boxes"][0]], ['Pre-Kriging', 'Post-Kriging'], loc='upper right')
#plt.savefig(f'{DATA_ROOT}/EcosystemAnalysis/Models/Biomes/plots/sumatra_CCI-kriging-binned-residuals.png', dpi=1200, bbox_inches='tight')
plt.show()

In [ ]:
# Density plots of residuals pre and post kriging

fig, axs = plt.subplots(1, 2, figsize=(18, 8))  # 1 row, 2 columns

# Shared max for limits
ma = np.nanmax(sumatra_1) + 50
step = 50
ticks = np.arange(0, ma + step, step)

# --- First plot ---
mask1 = (np.isnan(sumatra_1)) | (np.isnan(pre_pred))
h1 = axs[0].hist2d(sumatra_1[~mask1].squeeze(), pre_pred[~mask1].squeeze(), bins=(100, 100), cmap='Greens', norm=LogNorm())
cbar1 = fig.colorbar(h1[3], ax=axs[0])
cbar1.ax.tick_params(labelsize=14)
cbar1.ax.set_ylabel('Number of samples', fontsize=16)

axs[0].plot([0, ma], [0, ma], 'k--')
axs[0].set_xlim((0, ma))
axs[0].set_ylim((0, ma))
axs[0].set_xlabel('GEDI reference AGBD [Mg/ha]', fontsize=16)
axs[0].set_ylabel('Predicted AGB [Mg/ha]', fontsize=16)
axs[0].set_xticks(ticks)
axs[0].set_yticks(ticks)
axs[0].tick_params(labelsize=14)
axs[0].grid()
axs[0].set_aspect('equal')

# --- Second plot (example, change inputs as needed) ---
mask2 = (np.isnan(sumatra_1)) | (np.isnan(post_pred))
h2 = axs[1].hist2d(sumatra_1[~mask2].squeeze(), post_pred[~mask2].squeeze(), bins=(100, 100), cmap='Greens', norm=LogNorm())

cbar2 = fig.colorbar(h2[3], ax=axs[1])
cbar2.ax.tick_params(labelsize=14)
cbar2.ax.set_ylabel('Number of samples', fontsize=16)

axs[1].plot([0, ma], [0, ma], 'k--')
axs[1].set_xlim((0, ma))
axs[1].set_ylim((0, ma))
axs[1].set_xlabel('GEDI reference AGBD [Mg/ha]', fontsize=16)
axs[1].set_ylabel('Predicted AGB [Mg/ha]', fontsize=16)
axs[1].set_xticks(ticks)
axs[1].set_yticks(ticks)
axs[1].tick_params(labelsize=14)
axs[1].grid()
axs[1].set_aspect('equal')

plt.tight_layout()
#plt.savefig('vis/CCI-density.png', dpi=1200)
plt.show()

## For our predictions, post-kriging on GEDI evaluated on field

In [ ]:
composites = True
downsampled = True
kriging = True
ood = False
stripe_size = 50
num_footprints = 2500
lengthscale = 'dynamic'

model = '17997535-1_17997535-2_17997535-3'

suffix = '_composite' if composites else ''

if not kriging : post_kriging_path = f'predictions/nico_film/2020/{model}/kriging-sumatra_gedi{suffix}-gedi_50.tif'
else: post_kriging_path = f"Data/merged/{model}/merged_downsampled-{resolution}m_{stripe_size}_{num_footprints}{'_dynamic' if lengthscale == 'dynamic' else ''}{'_ood' if ood else ''}.tif"

In [ ]:
with rs.open(f'Data/merged/{model}/merged_downsampled-{resolution}m{suffix}.tif', 'r') as src1, \
    rs.open(post_kriging_path, 'r') as src2, \
    rs.open(f'{DATA_ROOT}/Sumatra-AGB/pred_rasters/agbd_{resolution}m.tif', 'r') as src3 :

    pre_pred = src1.read(1, out_shape = (1, sumatra_1.shape[0], sumatra_1.shape[1]), resampling = Resampling.bilinear)
    rescale_transform = src1.transform * src1.transform.scale((src1.width / pre_pred.shape[-1]), (src1.height / pre_pred.shape[-2]))

    post_pred = src2.read(1)
    if kriging: post_std = src2.read(2)
    else: post_std = src2.read(3)
    sumatra = src3.read(1)

    transform_corr_merged = src2.transform
    crs_corr_merged = src2.crs
    transform_sumatra = src3.transform
    crs_sumatra = src3.crs    

In [ ]:
post_std[np.isnan(post_pred)] = np.nan

In [ ]:
# Load the test split

with open('splits/splits-sumatra_gedi.pkl', 'rb') as f: split_map = pickle.load(f)

# Evaluate on the test set
# test_mask = (split_map != 1) & (split_map != 2) # exclude train and val

# actually, use all points with valid data
# test_mask = (split_map == 1) | (split_map == 2) | (split_map == 3)

valid_mask = (~np.isnan(sumatra_1)) & (~np.isnan(pre_pred)) & (~np.isnan(post_pred))
# test_mask = test_mask & valid_mask
test_mask = valid_mask

pre_residuals = pre_pred[test_mask] - sumatra_1[test_mask]
post_residuals = post_pred[test_mask] - sumatra_1[test_mask]

rmse_pre = np.sqrt(np.nanmean(pre_residuals**2))
rmse_post = np.sqrt(np.nanmean(post_residuals**2))

mae_pre = np.nanmean(np.abs(pre_residuals))
mae_post = np.nanmean(np.abs(post_residuals))

r2_pre = r2_score(sumatra_1[test_mask], pre_pred[test_mask])
r2_post = r2_score(sumatra_1[test_mask], post_pred[test_mask])

me_pre = np.nanmean(pre_residuals)
me_post = np.nanmean(post_residuals)

# print in two columns
print(f"{'Metric':<10} {'Pre-Kriging':<15} {'Post-Kriging':<15}")
print(f"{'-'*40}")
print(f"{'RMSE':<10} {rmse_pre:<15.2f} {rmse_post:<15.2f}")
print(f"{'MAE':<10} {mae_pre:<15.2f} {mae_post:<15.2f}")
print(f"{'ME':<10} {me_pre:<15.2f} {me_post:<15.2f}")
print(f"{'R2':<10} {r2_pre:<15.4f} {r2_post:<15.4f}")


In [ ]:
from copy import deepcopy
ours_post_pred_10m = deepcopy(post_pred)

In [ ]:
import matplotlib.pyplot as plt

VMIN, VMAX = 0, 250
CMAP = "viridis"
DPI = 300
row_start, row_end = 500, 1000
col_start, col_end = 550, 1050

panels = {
    "figs/ours_basemap.png": pre_pred,
    "figs/ours_basemap-zoomed.png": pre_pred[row_start:row_end, col_start:col_end],
    "figs/ours_post10m.png": post_pred,
    "figs/ours_post10m-zoomed.png": post_pred[row_start:row_end, col_start:col_end],
    # "figs/ours_post100m.png": post_pred,
    # "figs/ours_post100m-zoomed.png": post_pred[row_start:row_end, col_start:col_end],
    # "figs/esa_basemap.png": pre_pred,
    # "figs/esa_basemap-zoomed.png": pre_pred[row_start:row_end, col_start:col_end],
    # "figs/esa_post.png": post_pred,
    # "figs/esa_post-zoomed.png": post_pred[row_start:row_end, col_start:col_end],
    "figs/reference.png": sumatra_1,
    "figs/reference-zoomed.png": sumatra_1[row_start:row_end, col_start:col_end]
}

for path, data in panels.items():
    fig, ax = plt.subplots(figsize=(5, 5), constrained_layout=True)
    ax.imshow(data, cmap=CMAP, vmin=VMIN, vmax=VMAX)
    ax.axis("off")
    fig.savefig(path, dpi=DPI, bbox_inches="tight", pad_inches=0, transparent=True)
    plt.close(fig)
    print(f"Saved {path}")

In [ ]:
f, axarr = plt.subplots(1, 3, figsize=(15, 15), constrained_layout=True)
im1 = axarr[0].imshow(pre_pred, cmap='viridis', vmin=0, vmax=250)
axarr[0].set_title('Pre-Kriging AGB Density [t/ha]')
axarr[0].set_xticks([])  # Remove x-axis ticks
axarr[0].set_yticks([])  # Remove y-axis ticks
axarr[0].axis('off')
im2 = axarr[1].imshow(post_pred, cmap='viridis', vmin=0, vmax=250)
axarr[1].set_title('Post-Kriging AGB Density [t/ha]')
axarr[1].set_xticks([])  # Remove x-axis ticks
axarr[1].set_yticks([])  # Remove y-axis ticks
axarr[1].axis('off')
im3 = axarr[2].imshow(sumatra_1, cmap='viridis', vmin=0, vmax=250)
axarr[2].set_title('Reference AGB Density [t/ha]')
axarr[2].set_xticks([])  # Remove x-axis ticks
axarr[2].set_yticks([])  # Remove y-axis ticks
axarr[2].axis('off')

cbar = f.colorbar(im3, ax=axarr, shrink=0.2, orientation='vertical')
cbar.set_label('AGB Density [t/ha]')

plt.savefig('figs/ours-gedi-field.png', dpi=1200, bbox_inches='tight')
plt.show()

In [ ]:
# zoomed in version

row_start, row_end = 500, 1000
col_start, col_end = 550, 1050

f, axarr = plt.subplots(1, 3, figsize=(15, 15), constrained_layout=True)
im1 = axarr[0].imshow(pre_pred[row_start:row_end, col_start:col_end], cmap='viridis', vmin=0, vmax=250)
axarr[0].set_title('Pre-Kriging AGB Density [t/ha]')
axarr[0].set_xticks([])  # Remove x-axis ticks
axarr[0].set_yticks([])  # Remove y-axis ticks
axarr[0].axis('off')
im2 = axarr[1].imshow(post_pred[row_start:row_end, col_start:col_end], cmap='viridis', vmin=0, vmax=250)
axarr[1].set_title('Post-Kriging AGB Density [t/ha]')
axarr[1].set_xticks([])  # Remove x-axis ticks
axarr[1].set_yticks([])  # Remove y-axis ticks
axarr[1].axis('off')
im3 = axarr[2].imshow(sumatra_1[row_start:row_end, col_start:col_end], cmap='viridis', vmin=0, vmax=250)
axarr[2].set_title('Reference AGB Density [t/ha]')
axarr[2].set_xticks([])  # Remove x-axis ticks
axarr[2].set_yticks([])  # Remove y-axis ticks
axarr[2].axis('off')

cbar = f.colorbar(im3, ax=axarr, shrink=0.2, orientation='vertical')
cbar.set_label('AGB Density [t/ha]')

plt.savefig('figs/ours-gedi-field-zoom.png', dpi=1200, bbox_inches='tight')
plt.show()

In [ ]:
f, axarr = plt.subplots(1, 2, figsize=(15, 15), constrained_layout=True)

im1 = axarr[0].imshow(post_pred, cmap='viridis', vmin=0, vmax=250)
axarr[0].set_title('Post-Kriging AGB Density [t/ha]')
axarr[0].set_xticks([])  # Remove x-axis ticks
axarr[0].set_yticks([])  # Remove y-axis ticks
axarr[0].axis('off')

# add colorbar for im1
cbar1 = f.colorbar(im1, ax=axarr[0], shrink=0.6, orientation='vertical')
cbar1.set_label('AGB Density [t/ha]')

im2 = axarr[1].imshow(post_std, cmap='viridis', vmin=0, vmax=50)
axarr[1].set_title('Post-Kriging Standard Deviation [t/ha]')
axarr[1].set_xticks([])  # Remove x-axis ticks
axarr[1].set_yticks([])  # Remove y-axis ticks
axarr[1].axis('off')

# add colorbar for im2
cbar2 = f.colorbar(im2, ax=axarr[1], shrink=0.6, orientation='vertical')
cbar2.set_label('AGB Standard Deviation [t/ha]')

plt.show()

In [ ]:
# Binned residuas pre and post kriging

bins = np.arange(0, 300, 50)
lbs, ubs = bins[:-1], bins[1:]
labels = [f"{lb}-{ub}" for lb, ub in zip(lbs, ubs)]


pre_binned_residuals = [pre_residuals[(sumatra_1[test_mask] >= lb) & (sumatra_1[test_mask] < ub)] for lb, ub in zip(lbs, ubs)]
post_binned_residuals = [post_residuals[(sumatra_1[test_mask] >= lb) & (sumatra_1[test_mask] < ub)] for lb, ub in zip(lbs, ubs)]

fig, ax = plt.subplots(figsize=(10, 6))
box1 = ax.boxplot(pre_binned_residuals, showfliers=False, positions=bins[:-1] + 10, widths=20, patch_artist=True,
                  boxprops=dict(facecolor='lightblue', color='blue'),
                  medianprops=dict(color='darkblue'),
                  whiskerprops=dict(color='blue'),
                  capprops=dict(color='blue'),
                  flierprops=dict(color='blue', markeredgecolor='blue'))
box2 = ax.boxplot(post_binned_residuals, showfliers=False, positions=bins[:-1] + 30, widths=20, patch_artist=True,
                  boxprops=dict(facecolor='lightgreen', color='green'),
                  medianprops=dict(color='darkgreen'),
                  whiskerprops=dict(color='green'),
                  capprops=dict(color='green'),
                  flierprops=dict(color='green', markeredgecolor='green'))
ax.axhline(0, color='red', linestyle='--')
ax.set_xticks(bins[:-1] + 20)
ax.set_xticklabels(labels)
ax.set_ylim(-250,300)
ax.set_xlabel('AGB Density Bins [t/ha]')
ax.set_ylabel('Residuals [t/ha]')
ax.legend([box1["boxes"][0], box2["boxes"][0]], ['Pre-Kriging', 'Post-Kriging'], loc='upper left')
plt.savefig('figs/ours-gedi-field-binned-residuals.png', dpi=1200, bbox_inches='tight')
plt.show()

In [ ]:
# Density plots of residuals pre and post kriging

fig, axs = plt.subplots(1, 2, figsize=(18, 8))  # 1 row, 2 columns

# Shared max for limits
ma = np.nanmax(sumatra_1) + 50
ma = 350
step = 50
ticks = np.arange(0, ma + step, step)

# --- First plot ---
mask1 = (np.isnan(sumatra_1)) | (np.isnan(pre_pred))
h1 = axs[0].hist2d(sumatra_1[~mask1].squeeze(), pre_pred[~mask1].squeeze(), bins=(100, 100), cmap='Greens', norm=LogNorm(vmin=1, vmax=10000))
cbar1 = fig.colorbar(h1[3], ax=axs[0])
cbar1.ax.tick_params(labelsize=14)
cbar1.ax.set_ylabel('Number of samples', fontsize=16)

axs[0].plot([0, ma], [0, ma], 'k--')
axs[0].set_xlim((0, ma))
axs[0].set_ylim((0, ma))
axs[0].set_xlabel('GEDI reference AGBD [Mg/ha]', fontsize=16)
axs[0].set_ylabel('Predicted AGB [Mg/ha]', fontsize=16)
axs[0].set_xticks(ticks)
axs[0].set_yticks(ticks)
axs[0].tick_params(labelsize=14)
axs[0].grid()
axs[0].set_aspect('equal')


# --- Second plot (example, change inputs as needed) ---
mask2 = (np.isnan(sumatra_1)) | (np.isnan(post_pred))
h2 = axs[1].hist2d(sumatra_1[~mask2].squeeze(), post_pred[~mask2].squeeze(), bins=(100, 100), cmap='Greens', norm=LogNorm(vmin=1, vmax=10000))

cbar2 = fig.colorbar(h2[3], ax=axs[1])
cbar2.ax.tick_params(labelsize=14)
cbar2.ax.set_ylabel('Number of samples', fontsize=16)

axs[1].plot([0, ma], [0, ma], 'k--')
axs[1].set_xlim((0, ma))
axs[1].set_ylim((0, ma))
axs[1].set_xlabel('GEDI reference AGBD [Mg/ha]', fontsize=16)
axs[1].set_ylabel('Predicted AGB [Mg/ha]', fontsize=16)
axs[1].set_xticks(ticks)
axs[1].set_yticks(ticks)
axs[1].tick_params(labelsize=14)
axs[1].grid()
axs[1].set_aspect('equal')

plt.tight_layout()
plt.savefig('figs/ours-gedi-field-density.png', dpi=1200)
plt.show()

In [ ]:
# Save them individually

CMAP = 'Greens'
VMIN, VMAX = 1, 10000
DPI = 300
MA = 350
STEP = 50
TICKS = np.arange(0, MA + STEP, STEP)

panels = {
    "figs/scatter_ours_basemap.png":    (sumatra_1, pre_pred),
    "figs/scatter_ours_10m.png":        (sumatra_1, post_pred),
    # "figs/scatter_ours_100m.png":     (sumatra_1, post_pred),
    # "figs/scatter_esa_basemap.png":   (sumatra_1, pre_pred),
    # "figs/scatter_esa_post.png": (sumatra_1, post_pred),
}

for path, (ref, pred) in panels.items():
    fig, ax = plt.subplots(figsize=(6, 6))

    mask = np.isnan(ref) | np.isnan(pred)
    ax.hist2d(ref[~mask].squeeze(), pred[~mask].squeeze(),
              bins=(100, 100), cmap=CMAP, norm=LogNorm(vmin=VMIN, vmax=VMAX))

    ax.plot([0, MA], [0, MA], 'k--')
    ax.set_xlim((0, MA))
    ax.set_ylim((0, MA))
    ax.set_xticks(TICKS)
    ax.set_yticks(TICKS)
    ax.tick_params(labelsize=14)
    ax.grid()
    ax.set_aspect('equal')

    # No axis labels, no colorbar — compose script adds those
    fig.savefig(path, dpi=DPI, bbox_inches='tight', pad_inches=0.05)
    plt.close(fig)
    print(f"Saved {path}")

## For our predictions downsampled, post kriging on GEDI evaluated on field

In [ ]:
resolution = 100

suffix = '' if resolution == 100 else f'-{resolution}m'

with rs.open(f'Data/merged/17997535-1_17997535-2_17997535-3/merged_downsampled-100m_composite.tif', 'r') as src1, \
    rs.open(f'predictions/nico_film/2020/17997535-1_17997535-2_17997535-3/kriging-sumatra_downsampled_gedi_composite-gedi_50{suffix}.tif', 'r') as src2, \
    rs.open(f'{DATA_ROOT}/Sumatra-AGB/pred_rasters/agbd_{resolution}m.tif', 'r') as src3 :

    sumatra_1 = src3.read(1)
    
    pre_pred = src1.read(1, out_shape = (1, sumatra_1.shape[0], sumatra_1.shape[1]), resampling = Resampling.bilinear)
    rescale_transform = src1.transform * src1.transform.scale((src1.width / pre_pred.shape[-1]), (src1.height / pre_pred.shape[-2]))

    post_pred = src2.read(1)
    

    transform_corr_merged = src2.transform
    crs_corr_merged = src2.crs
    transform_sumatra = src3.transform
    crs_sumatra = src3.crs    



In [ ]:
# Load the test split

# with open('splits/splits-sumatra_cci_gedi.pkl', 'rb') as f: split_map = pickle.load(f)
# Evaluate on the test set
# test_mask = (split_map != 1) & (split_map != 2)
# test_mask = (split_map == 1) | (split_map == 2) | (split_map == 3)

valid_mask = (~np.isnan(sumatra_1)) & (~np.isnan(pre_pred)) & (~np.isnan(post_pred))
# test_mask = test_mask & valid_mask
test_mask = valid_mask

pre_residuals = pre_pred[test_mask] - sumatra_1[test_mask]
post_residuals = post_pred[test_mask] - sumatra_1[test_mask]

rmse_pre = np.sqrt(np.nanmean(pre_residuals**2))
rmse_post = np.sqrt(np.nanmean(post_residuals**2))

mae_pre = np.nanmean(np.abs(pre_residuals))
mae_post = np.nanmean(np.abs(post_residuals))

r2_pre = r2_score(sumatra_1[test_mask], pre_pred[test_mask])
r2_post = r2_score(sumatra_1[test_mask], post_pred[test_mask])

me_pre = np.nanmean(pre_residuals)
me_post = np.nanmean(post_residuals)

# print in two columns
print(f"{'Metric':<10} {'Pre-Kriging':<15} {'Post-Kriging':<15}")
print(f"{'-'*40}")
print(f"{'RMSE':<10} {rmse_pre:<15.2f} {rmse_post:<15.2f}")
print(f"{'MAE':<10} {mae_pre:<15.2f} {mae_post:<15.2f}")
print(f"{'ME':<10} {me_pre:<15.2f} {me_post:<15.2f}")

In [ ]:
import matplotlib.pyplot as plt

VMIN, VMAX = 0, 250
CMAP = "viridis"
DPI = 300
row_start, row_end = 500, 1000
col_start, col_end = 550, 1050

panels = {
    #"figs/ours_basemap.png": pre_pred,
    #"figs/ours_basemap-zoomed.png": pre_pred[row_start:row_end, col_start:col_end],
    #"figs/ours_post10m.png": post_pred,
    #"figs/ours_post10m-zoomed.png": post_pred[row_start:row_end, col_start:col_end],
    "figs/ours_post100m.png": post_pred,
    "figs/ours_post100m-zoomed.png": post_pred[row_start:row_end, col_start:col_end],
    # "figs/esa_basemap.png": pre_pred,
    # "figs/esa_basemap-zoomed.png": pre_pred[row_start:row_end, col_start:col_end],
    # "figs/esa_post.png": post_pred,
    # "figs/esa_post-zoomed.png": post_pred[row_start:row_end, col_start:col_end],
    #"figs/reference.png": sumatra_1,
    #"figs/reference-zoomed.png": sumatra_1[row_start:row_end, col_start:col_end]
}

for path, data in panels.items():
    fig, ax = plt.subplots(figsize=(5, 5), constrained_layout=True)
    ax.imshow(data, cmap=CMAP, vmin=VMIN, vmax=VMAX)
    ax.axis("off")
    fig.savefig(path, dpi=DPI, bbox_inches="tight", pad_inches=0, transparent=True)
    plt.close(fig)
    print(f"Saved {path}")

In [ ]:
f, axarr = plt.subplots(1, 3, figsize=(15, 15), constrained_layout=True)
im1 = axarr[0].imshow(pre_pred, cmap='viridis', vmin=0, vmax=250)
axarr[0].set_title('Pre-Kriging AGB Density [t/ha]')
axarr[0].set_xticks([])  # Remove x-axis ticks
axarr[0].set_yticks([])  # Remove y-axis ticks
axarr[0].axis('off')
im2 = axarr[1].imshow(post_pred, cmap='viridis', vmin=0, vmax=250)
axarr[1].set_title('Post-Kriging AGB Density [t/ha]')
axarr[1].set_xticks([])  # Remove x-axis ticks
axarr[1].set_yticks([])  # Remove y-axis ticks
axarr[1].axis('off')
im3 = axarr[2].imshow(sumatra_1, cmap='viridis', vmin=0, vmax=250)
axarr[2].set_title('Reference AGB Density [t/ha]')
axarr[2].set_xticks([])  # Remove x-axis ticks
axarr[2].set_yticks([])  # Remove y-axis ticks
axarr[2].axis('off')

cbar = f.colorbar(im3, ax=axarr, shrink=0.2, orientation='vertical')
cbar.set_label('AGB Density [t/ha]')

plt.savefig('figs/ours-downsampled-gedi-field.png', dpi=1200, bbox_inches='tight')
plt.show()

In [ ]:
# zoomed in version

row_start, row_end = 500, 1000
col_start, col_end = 550, 1050

f, axarr = plt.subplots(1, 3, figsize=(15, 15), constrained_layout=True)
im1 = axarr[0].imshow(pre_pred[row_start:row_end, col_start:col_end], cmap='viridis', vmin=0, vmax=250)
axarr[0].set_title('Pre-Kriging AGB Density [t/ha]')
axarr[0].set_xticks([])  # Remove x-axis ticks
axarr[0].set_yticks([])  # Remove y-axis ticks
axarr[0].axis('off')
im2 = axarr[1].imshow(post_pred[row_start:row_end, col_start:col_end], cmap='viridis', vmin=0, vmax=250)
axarr[1].set_title('Post-Kriging AGB Density [t/ha]')
axarr[1].set_xticks([])  # Remove x-axis ticks
axarr[1].set_yticks([])  # Remove y-axis ticks
axarr[1].axis('off')
im3 = axarr[2].imshow(sumatra_1[row_start:row_end, col_start:col_end], cmap='viridis', vmin=0, vmax=250)
axarr[2].set_title('Reference AGB Density [t/ha]')
axarr[2].set_xticks([])  # Remove x-axis ticks
axarr[2].set_yticks([])  # Remove y-axis ticks
axarr[2].axis('off')

cbar = f.colorbar(im3, ax=axarr, shrink=0.2, orientation='vertical')
cbar.set_label('AGB Density [t/ha]')

plt.savefig('figs/ours-downsampled-gedi-field-zoom.png', dpi=1200, bbox_inches='tight')
plt.show()

In [ ]:
# Binned residuas pre and post kriging

bins = np.arange(0, 300, 50)
lbs, ubs = bins[:-1], bins[1:]

labels = [f"{lb}-{ub}" for lb, ub in zip(lbs, ubs)]

pre_binned_residuals = [pre_residuals[(sumatra_1[test_mask] >= lb) & (sumatra_1[test_mask] < ub)] for lb, ub in zip(lbs, ubs)]
post_binned_residuals = [post_residuals[(sumatra_1[test_mask] >= lb) & (sumatra_1[test_mask] < ub)] for lb, ub in zip(lbs, ubs)]

fig, ax = plt.subplots(figsize=(10, 6))
box1 = ax.boxplot(pre_binned_residuals, showfliers=False, positions=bins[:-1] + 10, widths=20, patch_artist=True,
                  boxprops=dict(facecolor='lightblue', color='blue'),
                  medianprops=dict(color='darkblue'),
                  whiskerprops=dict(color='blue'),
                  capprops=dict(color='blue'),
                  flierprops=dict(color='blue', markeredgecolor='blue'))
box2 = ax.boxplot(post_binned_residuals, showfliers=False, positions=bins[:-1] + 30, widths=20, patch_artist=True,
                  boxprops=dict(facecolor='lightgreen', color='green'),
                  medianprops=dict(color='darkgreen'),
                  whiskerprops=dict(color='green'),
                  capprops=dict(color='green'),
                  flierprops=dict(color='green', markeredgecolor='green'))
ax.axhline(0, color='red', linestyle='--')
ax.set_xticks(bins[:-1] + 20)
ax.set_xticklabels(labels)
ax.set_ylim(-250,300)
ax.set_xlabel('AGB Density Bins [t/ha]')
ax.set_ylabel('Residuals [t/ha]')
ax.legend([box1["boxes"][0], box2["boxes"][0]], ['Pre-Kriging', 'Post-Kriging'], loc='upper left')
plt.savefig('figs/ours-downsampled-gedi-field-binned-residuals.png', dpi=1200, bbox_inches='tight')
plt.show()

In [ ]:
# Plotting post (10m) and post (100m) binned residuals

# post_pred and ours_post_pred_10m

# Binned residuas pre and post kriging

bins = np.arange(0, 300, 50)
lbs, ubs = bins[:-1], bins[1:]

labels = [f"{lb}-{ub}" for lb, ub in zip(lbs, ubs)]

pre_binned_residuals = [pre_residuals[(sumatra_1[test_mask] >= lb) & (sumatra_1[test_mask] < ub)] for lb, ub in zip(lbs, ubs)]
post_binned_residuals = [post_residuals[(sumatra_1[test_mask] >= lb) & (sumatra_1[test_mask] < ub)] for lb, ub in zip(lbs, ubs)]
post_binned_residuals_10m = [ours_post_pred_10m[test_mask][(sumatra_1[test_mask] >= lb) & (sumatra_1[test_mask] < ub)] - sumatra_1[test_mask][(sumatra_1[test_mask] >= lb) & (sumatra_1[test_mask] < ub)] for lb, ub in zip(lbs, ubs)]

fig, ax = plt.subplots(figsize=(10, 6))

box1 = ax.boxplot(pre_binned_residuals, showfliers=False, positions=bins[:-1] + 10, widths=10, patch_artist=True,
                  boxprops=dict(facecolor='#56B4E9', color='#0072B2'),
                  medianprops=dict(color='#004E7C'),
                  whiskerprops=dict(color='#0072B2'),
                  capprops=dict(color='#0072B2'),
                  flierprops=dict(color='#0072B2', markeredgecolor='#0072B2'))

box2 = ax.boxplot(post_binned_residuals_10m, showfliers=False, positions=bins[:-1] + 20, widths=10, patch_artist=True,
                  boxprops=dict(facecolor='#F5C862', color='#E69F00'),
                  medianprops=dict(color='#B37A00'),
                  whiskerprops=dict(color='#E69F00'),
                  capprops=dict(color='#E69F00'),
                  flierprops=dict(color='#E69F00', markeredgecolor='#E69F00'))

box3 = ax.boxplot(post_binned_residuals, showfliers=False, positions=bins[:-1] + 30, widths=10, patch_artist=True,
                  boxprops=dict(facecolor='#5EC8A5', color='#009E73'),
                  medianprops=dict(color='#006B4F'),
                  whiskerprops=dict(color='#009E73'),
                  capprops=dict(color='#009E73'),
                  flierprops=dict(color='#009E73', markeredgecolor='#009E73'))

ax.axhline(0, color='grey', linestyle='--', alpha=0.7, linewidth=1)
ax.set_xticks(bins[:-1] + 20)
ax.set_xticklabels(labels)
ax.set_ylim(-250,300)
ax.set_xlabel('AGB Density Bins [t/ha]')
ax.set_ylabel('Residuals [t/ha]')
ax.legend([box1["boxes"][0], box2["boxes"][0], box3["boxes"][0]], ['Pre-Kriging', 'Post-Kriging (10m)', 'Post-Kriging (100m)'], loc='upper left')
plt.savefig('figs/ours-both-gedi-field-binned-residuals.png', dpi=1200) #, bbox_inches='tight')
plt.show()

In [ ]:
# Density plots of residuals pre and post kriging

fig, axs = plt.subplots(1, 2, figsize=(18, 8))  # 1 row, 2 columns

# Shared max for limits
ma = np.nanmax(sumatra_1) + 50
ma = 350
step = 50
ticks = np.arange(0, ma + step, step)

# --- First plot ---
mask1 = (np.isnan(sumatra_1)) | (np.isnan(pre_pred))
h1 = axs[0].hist2d(sumatra_1[~mask1].squeeze(), pre_pred[~mask1].squeeze(), bins=(100, 100), cmap='Greens', norm=LogNorm(vmin=1, vmax=10000))
cbar1 = fig.colorbar(h1[3], ax=axs[0])
cbar1.ax.tick_params(labelsize=14)
cbar1.ax.set_ylabel('Number of samples', fontsize=16)

axs[0].plot([0, ma], [0, ma], 'k--')
axs[0].set_xlim((0, ma))
axs[0].set_ylim((0, ma))
axs[0].set_xlabel('GEDI reference AGBD [Mg/ha]', fontsize=16)
axs[0].set_ylabel('Predicted AGB [Mg/ha]', fontsize=16)
axs[0].set_xticks(ticks)
axs[0].set_yticks(ticks)
axs[0].tick_params(labelsize=14)
axs[0].grid()
axs[0].set_aspect('equal')

# --- Second plot (example, change inputs as needed) ---
mask2 = (np.isnan(sumatra_1)) | (np.isnan(post_pred))
h2 = axs[1].hist2d(sumatra_1[~mask2].squeeze(), post_pred[~mask2].squeeze(), bins=(100, 100), cmap='Greens', norm=LogNorm(vmin=1, vmax=10000))

cbar2 = fig.colorbar(h2[3], ax=axs[1])
cbar2.ax.tick_params(labelsize=14)
cbar2.ax.set_ylabel('Number of samples', fontsize=16)

axs[1].plot([0, ma], [0, ma], 'k--')
axs[1].set_xlim((0, ma))
axs[1].set_ylim((0, ma))
axs[1].set_xlabel('GEDI reference AGBD [Mg/ha]', fontsize=16)
axs[1].set_ylabel('Predicted AGB [Mg/ha]', fontsize=16)
axs[1].set_xticks(ticks)
axs[1].set_yticks(ticks)
axs[1].tick_params(labelsize=14)
axs[1].grid()
axs[1].set_aspect('equal')

plt.tight_layout()
plt.savefig('figs/ours-downsampled-gedi-field-density.png', dpi=1200)
plt.show()

In [ ]:
# Save them individually

CMAP = 'Greens'
VMIN, VMAX = 1, 10000
DPI = 300
MA = 350
STEP = 50
TICKS = np.arange(0, MA + STEP, STEP)

panels = {
    #"figs/scatter_ours_basemap.png":    (sumatra_1, pre_pred),
    #"figs/scatter_ours_10m.png":        (sumatra_1, post_pred),
    "figs/scatter_ours_100m.png":     (sumatra_1, post_pred),
    # "figs/scatter_esa_basemap.png":   (sumatra_1, pre_pred),
    # "figs/scatter_esa_post.png": (sumatra_1, post_pred),
}

for path, (ref, pred) in panels.items():
    fig, ax = plt.subplots(figsize=(6, 6))

    mask = np.isnan(ref) | np.isnan(pred)
    ax.hist2d(ref[~mask].squeeze(), pred[~mask].squeeze(),
              bins=(100, 100), cmap=CMAP, norm=LogNorm(vmin=VMIN, vmax=VMAX))

    ax.plot([0, MA], [0, MA], 'k--')
    ax.set_xlim((0, MA))
    ax.set_ylim((0, MA))
    ax.set_xticks(TICKS)
    ax.set_yticks(TICKS)
    ax.tick_params(labelsize=14)
    ax.grid()
    ax.set_aspect('equal')

    # No axis labels, no colorbar — compose script adds those
    fig.savefig(path, dpi=DPI, bbox_inches='tight', pad_inches=0.05)
    plt.close(fig)
    print(f"Saved {path}")

## CCI post kriging on GEDI evaluated on field

In [ ]:
resolution = 100

suffix = '' if resolution == 100 else f'-{resolution}m'

with rs.open(f'Data/CCI/CCI_N00E100{suffix}.tif', 'r') as src1, \
    rs.open(f'predictions/nico_film/2020/17997535-1_17997535-2_17997535-3/kriging-sumatra_cci_gedi-gedi_50{suffix}.tif', 'r') as src2, \
    rs.open(f'{DATA_ROOT}/Sumatra-AGB/pred_rasters/agbd_{resolution}m.tif', 'r') as src3 :

    sumatra_1 = src3.read(1)
    
    pre_pred = src1.read(1, out_shape = (1, sumatra_1.shape[0], sumatra_1.shape[1]), resampling = Resampling.bilinear)
    rescale_transform = src1.transform * src1.transform.scale((src1.width / pre_pred.shape[-1]), (src1.height / pre_pred.shape[-2]))

    post_pred = src2.read(1)
    

    transform_corr_merged = src2.transform
    crs_corr_merged = src2.crs
    transform_sumatra = src3.transform
    crs_sumatra = src3.crs    



In [ ]:
# Load the test split

# with open('splits/splits-sumatra_cci_gedi.pkl', 'rb') as f: split_map = pickle.load(f)
# Evaluate on the test set
# test_mask = (split_map != 1) & (split_map != 2)
# test_mask = (split_map == 1) | (split_map == 2) | (split_map == 3)

valid_mask = (~np.isnan(sumatra_1)) & (~np.isnan(pre_pred)) & (~np.isnan(post_pred))
# test_mask = test_mask & valid_mask
test_mask = valid_mask

pre_residuals = pre_pred[test_mask] - sumatra_1[test_mask]
post_residuals = post_pred[test_mask] - sumatra_1[test_mask]

rmse_pre = np.sqrt(np.nanmean(pre_residuals**2))
rmse_post = np.sqrt(np.nanmean(post_residuals**2))

mae_pre = np.nanmean(np.abs(pre_residuals))
mae_post = np.nanmean(np.abs(post_residuals))

r2_pre = r2_score(sumatra_1[test_mask], pre_pred[test_mask])
r2_post = r2_score(sumatra_1[test_mask], post_pred[test_mask])

me_pre = np.nanmean(pre_residuals)
me_post = np.nanmean(post_residuals)

# print in two columns
print(f"{'Metric':<10} {'Pre-Kriging':<15} {'Post-Kriging':<15}")
print(f"{'-'*40}")
print(f"{'RMSE':<10} {rmse_pre:<15.2f} {rmse_post:<15.2f}")
print(f"{'MAE':<10} {mae_pre:<15.2f} {mae_post:<15.2f}")
print(f"{'ME':<10} {me_pre:<15.2f} {me_post:<15.2f}")
print(f"{'R2':<10} {r2_pre:<15.4f} {r2_post:<15.4f}")

In [ ]:
import matplotlib.pyplot as plt

VMIN, VMAX = 0, 250
CMAP = "viridis"
DPI = 300
row_start, row_end = 500, 1000
col_start, col_end = 550, 1050

panels = {
    #"figs/ours_basemap.png": pre_pred,
    #"figs/ours_basemap-zoomed.png": pre_pred[row_start:row_end, col_start:col_end],
    #"figs/ours_post10m.png": post_pred,
    #"figs/ours_post10m-zoomed.png": post_pred[row_start:row_end, col_start:col_end],
    #"figs/ours_post100m.png": post_pred,
    #"figs/ours_post100m-zoomed.png": post_pred[row_start:row_end, col_start:col_end],
    "figs/esa_basemap.png": pre_pred,
    "figs/esa_basemap-zoomed.png": pre_pred[row_start:row_end, col_start:col_end],
    "figs/esa_post.png": post_pred,
    "figs/esa_post-zoomed.png": post_pred[row_start:row_end, col_start:col_end],
    #"figs/reference.png": sumatra_1,
    #"figs/reference-zoomed.png": sumatra_1[row_start:row_end, col_start:col_end]
}

for path, data in panels.items():
    fig, ax = plt.subplots(figsize=(5, 5), constrained_layout=True)
    ax.imshow(data, cmap=CMAP, vmin=VMIN, vmax=VMAX)
    ax.axis("off")
    fig.savefig(path, dpi=DPI, bbox_inches="tight", pad_inches=0, transparent=True)
    plt.close(fig)
    print(f"Saved {path}")

In [ ]:
f, axarr = plt.subplots(1, 3, figsize=(15, 15), constrained_layout=True)
im1 = axarr[0].imshow(pre_pred, cmap='viridis', vmin=0, vmax=250)
axarr[0].set_title('Pre-Kriging AGB Density [t/ha]')
axarr[0].set_xticks([])  # Remove x-axis ticks
axarr[0].set_yticks([])  # Remove y-axis ticks
axarr[0].axis('off')
im2 = axarr[1].imshow(post_pred, cmap='viridis', vmin=0, vmax=250)
axarr[1].set_title('Post-Kriging AGB Density [t/ha]')
axarr[1].set_xticks([])  # Remove x-axis ticks
axarr[1].set_yticks([])  # Remove y-axis ticks
axarr[1].axis('off')
im3 = axarr[2].imshow(sumatra_1, cmap='viridis', vmin=0, vmax=250)
axarr[2].set_title('Reference AGB Density [t/ha]')
axarr[2].set_xticks([])  # Remove x-axis ticks
axarr[2].set_yticks([])  # Remove y-axis ticks
axarr[2].axis('off')

cbar = f.colorbar(im3, ax=axarr, shrink=0.2, orientation='vertical')
cbar.set_label('AGB Density [t/ha]')

plt.savefig('figs/cci-gedi-field.png', dpi=1200, bbox_inches='tight')
plt.show()

In [ ]:
# zoomed in version

row_start, row_end = 500, 1000
col_start, col_end = 550, 1050

f, axarr = plt.subplots(1, 3, figsize=(15, 15), constrained_layout=True)
im1 = axarr[0].imshow(pre_pred[row_start:row_end, col_start:col_end], cmap='viridis', vmin=0, vmax=250)
axarr[0].set_title('Pre-Kriging AGB Density [t/ha]')
axarr[0].set_xticks([])  # Remove x-axis ticks
axarr[0].set_yticks([])  # Remove y-axis ticks
axarr[0].axis('off')
im2 = axarr[1].imshow(post_pred[row_start:row_end, col_start:col_end], cmap='viridis', vmin=0, vmax=250)
axarr[1].set_title('Post-Kriging AGB Density [t/ha]')
axarr[1].set_xticks([])  # Remove x-axis ticks
axarr[1].set_yticks([])  # Remove y-axis ticks
axarr[1].axis('off')
im3 = axarr[2].imshow(sumatra_1[row_start:row_end, col_start:col_end], cmap='viridis', vmin=0, vmax=250)
axarr[2].set_title('Reference AGB Density [t/ha]')
axarr[2].set_xticks([])  # Remove x-axis ticks
axarr[2].set_yticks([])  # Remove y-axis ticks
axarr[2].axis('off')

cbar = f.colorbar(im3, ax=axarr, shrink=0.2, orientation='vertical')
cbar.set_label('AGB Density [t/ha]')

plt.savefig('figs/cci-gedi-field-zoom.png', dpi=1200, bbox_inches='tight')
plt.show()

In [ ]:
# Binned residuas pre and post kriging

bins = np.arange(0, 300, 50)
lbs, ubs = bins[:-1], bins[1:]

labels = [f"{lb}-{ub}" for lb, ub in zip(lbs, ubs)]

pre_binned_residuals = [pre_residuals[(sumatra_1[test_mask] >= lb) & (sumatra_1[test_mask] < ub)] for lb, ub in zip(lbs, ubs)]
post_binned_residuals = [post_residuals[(sumatra_1[test_mask] >= lb) & (sumatra_1[test_mask] < ub)] for lb, ub in zip(lbs, ubs)]

fig, ax = plt.subplots(figsize=(10, 6))
box1 = ax.boxplot(pre_binned_residuals, showfliers=False, positions=bins[:-1] + 10, widths=10, patch_artist=True,
                  boxprops=dict(facecolor='#56B4E9', color='#0072B2'),
                  medianprops=dict(color='#004E7C'),
                  whiskerprops=dict(color='#0072B2'),
                  capprops=dict(color='#0072B2'),
                  flierprops=dict(color='#0072B2', markeredgecolor='#0072B2'))
box2 = ax.boxplot(post_binned_residuals, showfliers=False, positions=bins[:-1] + 20, widths=10, patch_artist=True,
                  boxprops=dict(facecolor='#5EC8A5', color='#009E73'),
                  medianprops=dict(color='#006B4F'),
                  whiskerprops=dict(color='#009E73'),
                  capprops=dict(color='#009E73'),
                  flierprops=dict(color='#009E73', markeredgecolor='#009E73'))
ax.axhline(0, color='red', linestyle='--')
ax.set_xticks(bins[:-1] + 20)
ax.set_xticklabels(labels)
ax.set_ylim(-250,300)
ax.set_xlabel('AGB Density Bins [t/ha]')
ax.set_ylabel('') # 'Residuals [t/ha]'
ax.set_yticklabels([])
ax.legend([box1["boxes"][0], box2["boxes"][0]], ['Pre-Kriging', 'Post-Kriging'], loc='upper left')
plt.savefig('figs/cci-gedi-field-binned-residuals.png', dpi=1200) #, bbox_inches='tight')
plt.show()

In [ ]:
# Density plots of residuals pre and post kriging

fig, axs = plt.subplots(1, 2, figsize=(18, 8))  # 1 row, 2 columns

# Shared max for limits
ma = np.nanmax(sumatra_1) + 50
ma = 350
step = 50
ticks = np.arange(0, ma + step, step)

# --- First plot ---
mask1 = (np.isnan(sumatra_1)) | (np.isnan(pre_pred))
h1 = axs[0].hist2d(sumatra_1[~mask1].squeeze(), pre_pred[~mask1].squeeze(), bins=(100, 100), cmap='Greens', norm=LogNorm(vmin=1, vmax=10000))
cbar1 = fig.colorbar(h1[3], ax=axs[0])
cbar1.ax.tick_params(labelsize=14)
cbar1.ax.set_ylabel('Number of samples', fontsize=16)

axs[0].plot([0, ma], [0, ma], 'k--')
axs[0].set_xlim((0, ma))
axs[0].set_ylim((0, ma))
axs[0].set_xlabel('GEDI reference AGBD [Mg/ha]', fontsize=16)
axs[0].set_ylabel('Predicted AGB [Mg/ha]', fontsize=16)
axs[0].set_xticks(ticks)
axs[0].set_yticks(ticks)
axs[0].tick_params(labelsize=14)
axs[0].grid()
axs[0].set_aspect('equal')

# --- Second plot (example, change inputs as needed) ---
mask2 = (np.isnan(sumatra_1)) | (np.isnan(post_pred))
h2 = axs[1].hist2d(sumatra_1[~mask2].squeeze(), post_pred[~mask2].squeeze(), bins=(100, 100), cmap='Greens', norm=LogNorm(vmin=1, vmax=10000))

cbar2 = fig.colorbar(h2[3], ax=axs[1])
cbar2.ax.tick_params(labelsize=14)
cbar2.ax.set_ylabel('Number of samples', fontsize=16)

axs[1].plot([0, ma], [0, ma], 'k--')
axs[1].set_xlim((0, ma))
axs[1].set_ylim((0, ma))
axs[1].set_xlabel('GEDI reference AGBD [Mg/ha]', fontsize=16)
axs[1].set_ylabel('Predicted AGB [Mg/ha]', fontsize=16)
axs[1].set_xticks(ticks)
axs[1].set_yticks(ticks)
axs[1].tick_params(labelsize=14)
axs[1].grid()
axs[1].set_aspect('equal')

plt.tight_layout()
plt.savefig('figs/cci-gedi-field-density.png', dpi=1200)
plt.show()

In [ ]:
# Save them individually

CMAP = 'Greens'
VMIN, VMAX = 1, 10000
DPI = 300
MA = 350
STEP = 50
TICKS = np.arange(0, MA + STEP, STEP)

panels = {
    #"figs/scatter_ours_basemap.png":    (sumatra_1, pre_pred),
    #"figs/scatter_ours_10m.png":        (sumatra_1, post_pred),
    #"figs/scatter_ours_100m.png":     (sumatra_1, post_pred),
    "figs/scatter_esa_basemap.png":   (sumatra_1, pre_pred),
    "figs/scatter_esa_post.png": (sumatra_1, post_pred),
}

for path, (ref, pred) in panels.items():
    fig, ax = plt.subplots(figsize=(6, 6))

    mask = np.isnan(ref) | np.isnan(pred)
    ax.hist2d(ref[~mask].squeeze(), pred[~mask].squeeze(),
              bins=(100, 100), cmap=CMAP, norm=LogNorm(vmin=VMIN, vmax=VMAX))

    ax.plot([0, MA], [0, MA], 'k--')
    ax.set_xlim((0, MA))
    ax.set_ylim((0, MA))
    ax.set_xticks(TICKS)
    ax.set_yticks(TICKS)
    ax.tick_params(labelsize=14)
    ax.grid()
    ax.set_aspect('equal')

    # No axis labels, no colorbar — compose script adds those
    fig.savefig(path, dpi=DPI, bbox_inches='tight', pad_inches=0.05)
    plt.close(fig)
    print(f"Saved {path}")

## Agreement of ours and CCI vs. GEDI, pre- and post-kriging

### Ours

In [ ]:
import geopandas as gpd
import pandas as pd

In [ ]:
# Base settings
composites = True
kriging = True
ood = False
stripe_size = 50
num_footprints = 2500
lengthscale = 'dynamic'
model = '17997535-1_17997535-2_17997535-3'

# Paths
suffix = '_composite' if composites else ''
if not kriging : post_kriging_path = f'predictions/nico_film/2020/{model}/kriging-sumatra_gedi{suffix}-gedi_50.tif'
else: post_kriging_path = f"Data/merged/{model}/merged_downsampled-{resolution}m_{stripe_size}_{num_footprints}{'_dynamic' if lengthscale == 'dynamic' else ''}{'_ood' if ood else ''}.tif"
path_gedi=f"{DATA_ROOT}/GEDI/Sumatra/L4A_Sumatra.gpkg"

# For each tile, load the test footprint indices
test_indices = []
for tile in ['47MRV', '48MTE', '48MUE', '47MRU', '48MTD', '48MUD', '47MRT', '48MTC', '48MUC'] :
    splits_path = f"splits-{tile}_0.3_0.2_{num_footprints}_15.0_{stripe_size}{'_ood' if ood else ''}-gedi-v2.pkl"
    with open(join('predictions', 'splits', splits_path), 'rb') as f: split_map_tile = pickle.load(f)['test']
    for _list in split_map_tile : test_indices.extend(_list)
print(f"Total test footprints: {len(test_indices)}")
print(f"Unique test footprints: {len(set(test_indices))}")

# Load pre- and post-kriging predictions
with rs.open(f'Data/merged/{model}/merged_downsampled-{resolution}m{suffix}.tif', 'r') as src1, rs.open(post_kriging_path, 'r') as src2 :
    pre_pred = src1.read(1)
    post_pred = src2.read(1)
    target_crs = src1.crs

# Load the GEDI footprints
GEDI = gpd.read_file(path_gedi, engine = 'pyogrio')
GEDI = GEDI.rename(columns={"index": "idx"})
GEDI = GEDI.to_crs(target_crs)

# Filter the GEDI footprints to the test set
GEDI = GEDI[GEDI['idx'].isin(test_indices)]

# Get the row and column indices of the GEDI footprints
with rs.open(post_kriging_path) as src:
    width, height = src.width, src.height
    def get_idx(geom, src) :
        lon, lat = geom.x, geom.y
        row_index, col_index = src.index(lon, lat)
        return row_index, col_index
    GEDI[['row_idx', 'col_idx']] = GEDI.apply(lambda row: get_idx(row['geometry'], src), axis = 1).apply(pd.Series)
GEDI = GEDI[(GEDI['row_idx'] < height) & (GEDI['row_idx'] >= 0) & (GEDI['col_idx'] < width) & (GEDI['col_idx'] >= 0)]

# If there are multiple footprints in the same pixel, aggregate them by median
GEDI = (GEDI.groupby(['row_idx', 'col_idx'], as_index=False).agg({'agbd': 'median', 'idx': list, **{col: 'first' for col in GEDI.select_dtypes(include='number').columns if col not in ['row_idx', 'col_idx', 'agbd', 'idx']}}))

# Get the predictions at the GEDI footprint locations
og_Y, og_X = GEDI['row_idx'].values.astype(int), GEDI['col_idx'].values.astype(int)
pre_agb = pre_pred[og_Y, og_X]
post_agb = post_pred[og_Y, og_X]
ref_agb = GEDI['agbd'].values

# Get only valid values
valid_mask = (~np.isnan(pre_agb)) & (~np.isnan(post_agb)) & (~np.isnan(ref_agb))
pre_agb = pre_agb[valid_mask]
post_agb = post_agb[valid_mask]
ref_agb = ref_agb[valid_mask]

# Compute the RMSE, MAE, and ME
pre_residuals = pre_agb - ref_agb
post_residuals = post_agb - ref_agb

pre_rmse = np.sqrt(np.mean(pre_residuals**2))
post_rmse = np.sqrt(np.mean(post_residuals**2))
pre_mae = np.mean(np.abs(pre_residuals))
post_mae = np.mean(np.abs(post_residuals))
pre_me = np.mean(pre_residuals)
post_me = np.mean(post_residuals)

# Print the results
print(f"{'Metric':<10} {'Pre-Kriging':<15} {'Post-Kriging':<15}")
print(f"{'-'*40}")
print(f"{'RMSE':<10} {pre_rmse:<15.2f} {post_rmse:<15.2f}")
print(f"{'MAE':<10} {pre_mae:<15.2f} {post_mae:<15.2f}")
print(f"{'ME':<10} {pre_me:<15.2f} {post_me:<15.2f}")


### CCI

In [ ]:
# Load the CCI pre- and post-kriging predictions
resolution = 100
suffix = '' if resolution == 100 else f'-{resolution}m'
post_kriging_path = f'predictions/nico_film/2020/17997535-1_17997535-2_17997535-3/kriging-sumatra_cci_gedi-gedi_50{suffix}.tif'
with rs.open(f'Data/CCI/CCI_N00E100{suffix}.tif', 'r') as src1, rs.open(post_kriging_path, 'r') as src2 :
    pre_pred = src1.read(1)
    post_pred = src2.read(1)
    target_crs = src1.crs

# Load the test footprints
test_indices = []
with open('predictions/splits/splits-sumatra_cci_gedi_composite_0.3_0.2_25000_15.0_50-gedi-v2.pkl', 'rb') as f: 
    split_map_tile = pickle.load(f)['test']
    for _list in split_map_tile : test_indices.extend(_list)
print(f"Total test footprints: {len(test_indices)}")
print(f"Unique test footprints: {len(set(test_indices))}")

# Load the GEDI footprints
GEDI = gpd.read_file(path_gedi, engine = 'pyogrio')
GEDI = GEDI.rename(columns={"index": "idx"})
GEDI = GEDI.to_crs(target_crs)

# Filter the GEDI footprints to the test set
GEDI = GEDI[GEDI['idx'].isin(test_indices)]

# Get the row and column indices of the GEDI footprints
with rs.open(post_kriging_path) as src:
    width, height = src.width, src.height
    def get_idx(geom, src) :
        lon, lat = geom.x, geom.y
        row_index, col_index = src.index(lon, lat)
        return row_index, col_index
    GEDI[['row_idx', 'col_idx']] = GEDI.apply(lambda row: get_idx(row['geometry'], src), axis = 1).apply(pd.Series)
GEDI = GEDI[(GEDI['row_idx'] < height) & (GEDI['row_idx'] >= 0) & (GEDI['col_idx'] < width) & (GEDI['col_idx'] >= 0)]

# If there are multiple footprints in the same pixel, aggregate them by median
GEDI = (GEDI.groupby(['row_idx', 'col_idx'], as_index=False).agg({'agbd': 'median', 'idx': list, **{col: 'first' for col in GEDI.select_dtypes(include='number').columns if col not in ['row_idx', 'col_idx', 'agbd', 'idx']}}))

# Get the predictions at the GEDI footprint locations
og_Y, og_X = GEDI['row_idx'].values.astype(int), GEDI['col_idx'].values.astype(int)
pre_agb = pre_pred[og_Y, og_X]
post_agb = post_pred[og_Y, og_X]
ref_agb = GEDI['agbd'].values

# Get only valid values
valid_mask = (~np.isnan(pre_agb)) & (~np.isnan(post_agb)) & (~np.isnan(ref_agb))
pre_agb = pre_agb[valid_mask]
post_agb = post_agb[valid_mask]
ref_agb = ref_agb[valid_mask]

# Compute the RMSE, MAE, and ME
pre_residuals = pre_agb - ref_agb
post_residuals = post_agb - ref_agb

pre_rmse = np.sqrt(np.mean(pre_residuals**2))
post_rmse = np.sqrt(np.mean(post_residuals**2))
pre_mae = np.mean(np.abs(pre_residuals))
post_mae = np.mean(np.abs(post_residuals))
pre_me = np.mean(pre_residuals)
post_me = np.mean(post_residuals)

# Print the results
print(f"{'Metric':<10} {'Pre-Kriging':<15} {'Post-Kriging':<15}")
print(f"{'-'*40}")
print(f"{'RMSE':<10} {pre_rmse:<15.2f} {post_rmse:<15.2f}")
print(f"{'MAE':<10} {pre_mae:<15.2f} {post_mae:<15.2f}")
print(f"{'ME':<10} {pre_me:<15.2f} {post_me:<15.2f}")
